# Transformer Components from Scratch

This notebook covers:
1. **Custom Multi-Head Self-Attention Modules**
2. **Vision Transformer (ViT) from Scratch** (PatchEmbedding + Class Token + Positional Embedding + TransformerEncoder)
3. **Custom BERT/RoBERTa Backbones** (Pure TransformerEncoder stacks with MLM heads)

All components are built from scratch or using minimal PyTorch utilities.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# =====================================================================
# MULTI-HEAD SELF-ATTENTION (CUSTOM IMPLEMENTATION)
# =====================================================================

class MultiHeadAttention(nn.Module):
    """
    Multi-Head Self-Attention mechanism.
    
    Allows the model to attend to information from different representation
    subspaces at different positions.
    
    Args:
        d_model: Model dimension
        n_heads: Number of attention heads
        dropout: Dropout probability
    """
    
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        
        # Linear projections for Q, K, V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        
        # Output projection
        self.W_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        self.scale = math.sqrt(self.head_dim)
    
    def split_heads(self, x):
        """Split heads: [batch, seq_len, d_model] -> [batch, n_heads, seq_len, head_dim]"""
        batch_size = x.shape[0]
        x = x.view(batch_size, -1, self.n_heads, self.head_dim)
        return x.transpose(1, 2)
    
    def forward(self, query, key, value, mask=None):
        """
        Args:
            query, key, value: [batch_size, seq_len, d_model]
            mask: [batch_size, 1, 1, seq_len]
        
        Returns:
            output: [batch_size, seq_len, d_model]
            attention_weights: [batch_size, n_heads, seq_len, seq_len]
        """
        batch_size = query.shape[0]
        
        # Linear projections
        Q = self.split_heads(self.W_q(query))  # [batch, n_heads, seq_len, head_dim]
        K = self.split_heads(self.W_k(key))
        V = self.split_heads(self.W_v(value))
        
        # Compute attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale  # [batch, n_heads, seq_len, seq_len]
        
        # Apply mask if provided
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        # Softmax to get attention weights
        attention_weights = F.softmax(scores, dim=-1)  # [batch, n_heads, seq_len, seq_len]
        attention_weights = self.dropout(attention_weights)
        
        # Apply attention to values
        context = torch.matmul(attention_weights, V)  # [batch, n_heads, seq_len, head_dim]
        
        # Concatenate heads
        context = context.transpose(1, 2).contiguous()
        context = context.view(batch_size, -1, self.d_model)
        
        # Final linear projection
        output = self.W_o(context)
        
        return output, attention_weights


In [ ]:
# =====================================================================
# FEED-FORWARD NETWORK
# =====================================================================

class PositionWiseFeedForward(nn.Module):
    """Position-Wise Feed-Forward Network (FFN)"""
    
    def __init__(self, d_model, d_ff=2048, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        return self.fc2(self.dropout(F.relu(self.fc1(x))))


# =====================================================================
# TRANSFORMER ENCODER LAYER
# =====================================================================

class TransformerEncoderLayer(nn.Module):
    """Single Transformer Encoder Layer"""
    
    def __init__(self, d_model, n_heads, d_ff=2048, dropout=0.1):
        super().__init__()
        
        # Multi-head attention
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        
        # Feed-forward
        self.ffn = PositionWiseFeedForward(d_model, d_ff, dropout)
        
        # Layer normalization and residual connections
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Self-attention with residual connection
        attn_output, _ = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        
        # Feed-forward with residual connection
        ffn_output = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_output))
        
        return x


In [ ]:
# =====================================================================
# POSITIONAL ENCODING
# =====================================================================

class PositionalEncoding(nn.Module):
    """Positional Encoding using sine and cosine functions"""
    
    def __init__(self, d_model, max_seq_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        # Create position encoding matrix
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                             (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe.unsqueeze(0))
    
    def forward(self, x):
        """x shape: [batch_size, seq_len, d_model]"""
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# =====================================================================
# PATCH EMBEDDING FOR VISION TRANSFORMER
# =====================================================================

class PatchEmbedding(nn.Module):
    """Convert image patches to embeddings using Conv2d"""
    
    def __init__(self, img_size, patch_size, in_channels, embed_dim):
        super().__init__()
        self.patch_size = patch_size
        self.n_patches = (img_size // patch_size) ** 2
        
        # Convolutional layer to extract patches
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
    
    def forward(self, x):
        """
        Args:
            x: [batch_size, channels, height, width]
        
        Returns:
            patches: [batch_size, n_patches, embed_dim]
        """
        x = self.proj(x)  # [batch, embed_dim, h', w']
        x = x.flatten(2)  # [batch, embed_dim, n_patches]
        x = x.transpose(1, 2)  # [batch, n_patches, embed_dim]
        return x


In [ ]:
# =====================================================================
# VISION TRANSFORMER (ViT) FROM SCRATCH
# =====================================================================

class VisionTransformer(nn.Module):
    """Vision Transformer: Image Patches + Positional Embedding + Transformer Stack"""
    
    def __init__(self, img_size, patch_size, in_channels, embed_dim, 
                 n_heads, n_layers, d_ff=2048, n_classes=10, dropout=0.1):
        super().__init__()
        
        # Patch embedding
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        n_patches = self.patch_embed.n_patches
        
        # Class token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        
        # Positional embedding for patches + class token
        self.pos_embedding = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))
        nn.init.trunc_normal_(self.pos_embedding, std=0.02)
        
        # Transformer encoder
        self.transformer = nn.Sequential(
            *[TransformerEncoderLayer(embed_dim, n_heads, d_ff, dropout) 
              for _ in range(n_layers)]
        )
        
        # Layer norm and classification head
        self.norm = nn.LayerNorm(embed_dim)
        self.cls_head = nn.Linear(embed_dim, n_classes)
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        """
        Args:
            x: [batch_size, channels, height, width]
        
        Returns:
            logits: [batch_size, n_classes]
        """
        batch_size = x.shape[0]
        
        # Extract patches and embed
        x = self.patch_embed(x)  # [batch, n_patches, embed_dim]
        
        # Prepend class token
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)  # [batch, 1, embed_dim]
        x = torch.cat([cls_tokens, x], dim=1)  # [batch, n_patches+1, embed_dim]
        
        # Add positional embedding
        x = x + self.pos_embedding
        x = self.dropout(x)
        
        # Apply transformer layers
        x = self.transformer(x)  # [batch, n_patches+1, embed_dim]
        
        # Layer norm and extract class token
        x = self.norm(x)
        x = x[:, 0, :]  # [batch, embed_dim]
        
        # Classification head
        logits = self.cls_head(x)  # [batch, n_classes]
        
        return logits


In [ ]:
# =====================================================================
# BERT/RoBERTa BACKBONE WITH MASKED LANGUAGE MODEL HEAD
# =====================================================================

class BertBackbone(nn.Module):
    """BERT-style Transformer backbone with token embeddings"""
    
    def __init__(self, vocab_size, embed_dim, n_heads, n_layers, 
                 d_ff=2048, max_seq_len=512, dropout=0.1):
        super().__init__()
        
        # Token embeddings
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        
        # Positional embeddings
        self.pos_embed = nn.Embedding(max_seq_len, embed_dim)
        
        # Segment embeddings (for sentence A/B separation)
        self.segment_embed = nn.Embedding(2, embed_dim)
        
        # Embedding layer norm and dropout
        self.norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)
        
        # Transformer encoder
        self.transformer = nn.Sequential(
            *[TransformerEncoderLayer(embed_dim, n_heads, d_ff, dropout)
              for _ in range(n_layers)]
        )
        
        self.embed_dim = embed_dim
    
    def forward(self, token_ids, segment_ids=None):
        """
        Args:
            token_ids: [batch_size, seq_len]
            segment_ids: [batch_size, seq_len]
        
        Returns:
            output: [batch_size, seq_len, embed_dim]
        """
        seq_len = token_ids.shape[1]
        positions = torch.arange(seq_len, device=token_ids.device).unsqueeze(0)
        
        # Token embeddings
        token_embeds = self.token_embed(token_ids)
        pos_embeds = self.pos_embed(positions)
        
        # Segment embeddings
        if segment_ids is None:
            segment_ids = torch.zeros_like(token_ids)
        seg_embeds = self.segment_embed(segment_ids)
        
        # Combine embeddings
        x = token_embeds + pos_embeds + seg_embeds
        x = self.norm(x)
        x = self.dropout(x)
        
        # Transformer
        x = self.transformer(x)
        
        return x


class MLMHead(nn.Module):
    """Masked Language Model Head for BERT pretraining"""
    
    def __init__(self, embed_dim, vocab_size):
        super().__init__()
        self.fc = nn.Linear(embed_dim, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size)
    
    def forward(self, x):
        x = self.fc(x)
        x = F.gelu(x)
        x = self.norm(x)
        logits = self.lm_head(x)
        return logits


class BertWithMLM(nn.Module):
    """BERT model with Masked Language Model head for pretraining"""
    
    def __init__(self, vocab_size, embed_dim, n_heads, n_layers, 
                 d_ff=2048, max_seq_len=512, dropout=0.1):
        super().__init__()
        self.backbone = BertBackbone(vocab_size, embed_dim, n_heads, n_layers,
                                     d_ff, max_seq_len, dropout)
        self.mlm_head = MLMHead(embed_dim, vocab_size)
    
    def forward(self, token_ids, segment_ids=None):
        x = self.backbone(token_ids, segment_ids)  # [batch, seq_len, embed_dim]
        logits = self.mlm_head(x)  # [batch, seq_len, vocab_size]
        return logits


In [ ]:
# =====================================================================
# DEMONSTRATION AND TESTING
# =====================================================================

if __name__ == "__main__":
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}\n")
    
    # Test 1: Vision Transformer
    print("=" * 60)
    print("VISION TRANSFORMER (ViT)")
    print("=" * 60)
    vit = VisionTransformer(
        img_size=224,
        patch_size=16,
        in_channels=3,
        embed_dim=768,
        n_heads=12,
        n_layers=12,
        d_ff=3072,
        n_classes=1000
    ).to(device)
    
    x_img = torch.randn(2, 3, 224, 224).to(device)
    vit.eval()
    with torch.no_grad():
        logits = vit(x_img)
    print(f"Input shape: {x_img.shape}")
    print(f"Output shape: {logits.shape}")
    print(f"✓ ViT working correctly!\n")
    
    # Test 2: BERT with MLM
    print("=" * 60)
    print("BERT WITH MASKED LANGUAGE MODEL")
    print("=" * 60)
    bert = BertWithMLM(
        vocab_size=30522,
        embed_dim=768,
        n_heads=12,
        n_layers=12,
        d_ff=3072
    ).to(device)
    
    token_ids = torch.randint(0, 30522, (4, 512)).to(device)
    bert.eval()
    with torch.no_grad():
        mlm_logits = bert(token_ids)
    print(f"Token IDs shape: {token_ids.shape}")
    print(f"MLM Output shape: {mlm_logits.shape}")
    print(f"✓ BERT with MLM working correctly!\n")
    
    print("✅ All Transformer components working!")
